In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import xarray as xr
import numpy as np
from typing import List, Optional, Sequence

In [ ]:
class SurfaceVarDiffPlotter:
    """
    Read preprocessed NetCDF files (from SurfaceVarDiffProcessor) and
    generate a 2xN multi-panel plot: Bias (top) and Spread (bottom).
    """

    def __init__(self, *, confidence_level: float = 0.05):
        self.confidence_level = confidence_level

    def _get_lat_lon_names(self, da: xr.DataArray):
        lon_name = next((k for k in da.coords if k.lower() in ["lon","longitude"]), None)
        lat_name = next((k for k in da.coords if k.lower() in ["lat","latitude"]), None)
        return lon_name, lat_name

    def plot_bias_and_stddev_multi_panel(self,
                                         processed_files: Sequence[str],
                                         model_labels: Optional[Sequence[str]] = None,
                                         bias_levels=None,
                                         spread_levels=None,
                                         cmap_bias="RdBu_r", cmap_std="viridis",
                                         figsize=(12,5), savepath=None, fontz=8):
        """
        processed_files: list of .nc paths (one per model case for the SAME variable/date)
        """
        ncols = len(processed_files)
        if model_labels is None:
            # derive from attrs if possible
            model_labels = []
            for fp in processed_files:
                attrs = xr.open_dataset(fp).attrs
                label = attrs.get("model_key", os.path.basename(fp).replace(".nc",""))
                model_labels.append(label)

        if bias_levels is None:
            bias_levels = [-5, -3, -1, -0.5, -0.1, 0.1, 0.5, 1, 3, 5]
        if spread_levels is None:
            spread_levels = [0.1, 0.5, 1, 1.5, 2, 2.5]

        bias_cmap   = plt.get_cmap(cmap_bias)
        spread_cmap = plt.get_cmap(cmap_std)
        bias_norm   = mcolors.BoundaryNorm(bias_levels,   bias_cmap.N)
        spread_norm = mcolors.BoundaryNorm(spread_levels, spread_cmap.N)

        fig, axes = plt.subplots(2, ncols, figsize=figsize,
                                 subplot_kw={"projection": ccrs.PlateCarree()})

        # plot panels
        pcc_rmse_texts = []
        for j, (fp, label) in enumerate(zip(processed_files, model_labels)):
            ds = xr.open_dataset(fp)
            bias = ds["bias"]
            std  = ds["model_std"]
            obs  = ds["obs_field"]
            sig  = ds["sig_mask"]

            # coordinates
            lon_name_b, lat_name_b = self._get_lat_lon_names(bias)
            lon_name_s, lat_name_s = self._get_lat_lon_names(std)

            # bias panel
            ax0 = axes[0, j]
            cf0 = ax0.contourf(bias[lon_name_b], bias[lat_name_b], bias,
                               levels=bias_levels, cmap=bias_cmap, norm=bias_norm,
                               transform=ccrs.PlateCarree(), extend="both")
            ax0.coastlines(linewidth=0.4)
            ax0.set_title(f"{label} - {ds.attrs.get('obs_key','OBS')} (Bias)", fontsize=fontz)
            ax0.set_xticks(np.arange(-180, 181, 60), crs=ccrs.PlateCarree())
            ax0.set_yticks(np.arange(-90, 91, 30),  crs=ccrs.PlateCarree())
            ax0.tick_params(labelsize=fontz-2)
            ax0.set_xlabel("Longitude", fontsize=fontz-1)
            ax0.set_ylabel("Latitude",  fontsize=fontz-1)

            # hatching for significant bias (p<alpha)
            sig.plot.contourf(ax=ax0, transform=ccrs.PlateCarree(),
                              colors='none', hatches=['..'], add_colorbar=False, add_labels=False)

            # annotate global PCC/RMSE
            pcc = float(ds.attrs.get("pcc_global", np.nan))
            rmse = float(ds.attrs.get("rmse_global", np.nan))
            ax0.text(0.98, 0.02, f"PCC: {pcc:.2f}\nRMSE: {rmse:.2f}",
                     transform=ax0.transAxes, fontsize=fontz-2, va='bottom', ha='right',
                     bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.1'))

            # spread panel
            ax1 = axes[1, j]
            cf1 = ax1.contourf(std[lon_name_s], std[lat_name_s], std,
                               levels=spread_levels, cmap=spread_cmap, norm=spread_norm,
                               transform=ccrs.PlateCarree(), extend="both")
            ax1.coastlines(linewidth=0.4)
            ax1.set_title(f"{label} Spread", fontsize=fontz)
            ax1.set_xticks(np.arange(-180, 181, 60), crs=ccrs.PlateCarree())
            ax1.set_yticks(np.arange(-90, 91, 30),  crs=ccrs.PlateCarree())
            ax1.tick_params(labelsize=fontz-2)
            ax1.set_xlabel("Longitude", fontsize=fontz-1)
            ax1.set_ylabel("Latitude",  fontsize=fontz-1)

        # shared colorbars on the right
        plt.subplots_adjust(wspace=0.1, hspace=0.2, right=0.88)

        cbar_ax1 = fig.add_axes([0.90, 0.60, 0.012, 0.30])
        sm1 = mpl.cm.ScalarMappable(norm=bias_norm, cmap=bias_cmap)
        sm1.set_array([])
        fig.colorbar(sm1, cax=cbar_ax1, label="Bias", extend='both')

        cbar_ax2 = fig.add_axes([0.90, 0.18, 0.012, 0.30])
        sm2 = mpl.cm.ScalarMappable(norm=spread_norm, cmap=spread_cmap)
        sm2.set_array([])
        fig.colorbar(sm2, cax=cbar_ax2, label="Spread", extend='both')

        if savepath:
            fig.savefig(savepath, dpi=600, bbox_inches='tight')
            print(f"[SAVED] Figure saved to: {savepath}")
        plt.show()


In [ ]:
if __name__ == "__main__":
    # Paths
    top_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    out_path = os.path.join(top_path, "diag_dart")
    fig_path = "./figures"
    os.makedirs(fig_path, exist_ok=True)

    landmask_file = os.path.join(out_path, "landmask_1x1.nc")

    # Data sources
    data_dict = {
        'GPCP': {
            'path': f'/compyfs/zhan391/acme_init/Observations/GPCP/daily',
            'template': 'PRECT.daily.%(year)s.nc',
            'frequency': 'daily',
            'nens': 1
        },
        'ERA5': {
            'path': f'{top_path}/Observations/ERA5/6hourly',
            'template': 'ERA5.6hourly.en00.TREFHT.%(year)s01-%(year)s12.nc',
            'frequency': '6hourly',
            'nens': 1
        },
        'CTRLEN10': {
            'path': f'{top_path}/CTRLEN10_15day_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily',
            'template': 'PRECT.%(ensemble)s.%(year)s.nc',
            'frequency': 'daily',
            'nens': 10
        },
        'CAPTEN10': {
            'path': f'{top_path}/CAPTEN10_15day_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily',
            'template': 'PRECT.%(ensemble)s.%(year)s.nc',
            'frequency': 'daily',
            'nens': 10
        },
        'DARTEN20': {
            'path': f'{top_path}/DARTEN20_15day_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily',
            'template': 'PRECT.%(ensemble)s.%(year)s.nc',
            'frequency': 'daily',
            'nens': 20
        },
        'DARTEN40': {
            'path': f'{top_path}/DARTEN40_15day_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily',
            'template': 'PRECT.%(ensemble)s.%(year)s.nc',
            'frequency': 'daily',
            'nens': 40
        }
    }

    # Configuration
    model_keys = ['CTRLEN10', 'CAPTEN10', 'DARTEN20', 'DARTEN40']
    obs_key = 'GPCP'
    obs_variable='PRECT'
    obs_fscale = 1.0
    variable='PRECT'
    fscale = 86400000.0 
    bias_levels = [-8,-5,-3,-2,-1,-0.1,0,0.1,1,2,3,5,8] 
    confidence_level = 0.05
    #np.linspace(-5,5,11) #[-1.0,-0.8,-0.6,-0.4,-0.2,-0.1,0.1,0.2,0.4,0.6,0.8,1.0]
    #spread_levels = [0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
    spread_levels = np.linspace(0,2,11)
    fontz = 18
    
    # Step 2: read the saved files and plot
    plotter = SurfaceVarDiffPlotter(confidence_level=confidence_level)
    plotter.plot_bias_and_stddev_multi_panel(
        processed_files=[out1, out2, out3],
        model_labels=["CAPTEN10", "DARTEN20", "DARTEN40"],
        figsize=(13,5),
        savepath="/path/fig_PRECT_bias_spread_2012-01-01.png",
        fontz=9
    )

    for target_date in [ '2012-01-01', '2012-01-05', '2012-01-10', '2012-01-15']:

        combined_savefile = os.path.join(
            fig_path, f"{variable}_bias_stddev_vs_{obs_key}_{target_date.replace('-', '')}.pdf"
        )
        
        plotter.plot_bias_and_stddev_multi_panel(
            processed_files=[out1, out2, out3],
            model_labels = model_labels,
            target_date=target_date,
            obs_key=obs_key,
            bias_levels=bias_levels,
            spread_levels=spread_levels,
            figsize = figsize,
            fontz=fontz,
            cmap_bias=cmocean.cm.curl,
            cmap_std=cmocean.cm.deep,
            savepath = combined_savefile,
        )

In [ ]:
if __name__ == "__main__":
    top_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    out_path = "/compyfs/zhan391/v3_dart_cda_scratch/diag_dart"
    fig_path = "/qfs/people/zhan391/e3sm_dart_work/analysis/diagnostic/figures/mjo_analysis"
    rmm_dir = "/compyfs/zhan391/acme_init/Observations/MJO_INDEX"
    compset = "F20TR"
    resolution = "ne30pg2_r05_IcoswISC30E3r5"
    machine = "compy"
    exp_base = f"{compset}_{resolution}_{machine}"

    os.makedirs(out_path, exist_ok=True)
    os.makedirs(fig_path, exist_ok=True)

    region_dict = {"Tropics": [-10, 10]}
    
    varname = 'PRECT'
    var_dict = {
        'PRECT': {'alias':'Precipitation', 'units': "mm/day"} 
    }
    varname = "PRECT"
    vmin    = -5
    vmax    = 5 
    nlevs   = 11
    
    frequency = "daily"
    time_start = "2011-12-01"
    time_end = "2012-02-29"
    
    model_list = list(data_dict.keys())

    analyzer = HovmollerPrecipAnalyzer(
        data_dict=data_dict,
        varname=varname,
        var_info=var_dict[varname],
        lat_bounds=region_dict[region],
        dt_hours=24,
        rmm_path=rmm_dir,
        time_start = time_start, 
        time_end = time_end
    )

    analyzer.plot_rmm_phase_space(
        start=time_start, 
        end=time_end,
        amp_thresh=1.0,
        savepath=os.path.join(
            fig_path, 
            "rmm_phase_space.pdf"
        )
    )
    
    for exp in model_list:
        print(f"\n[INFO] Running MJO analysis for {exp}")
        output_dir = os.path.join(fig_path, f"mjo_{exp}")
        os.makedirs(output_dir, exist_ok=True)
        
        comps = analyzer.compute_phase_composites(exp)
        figname = os.path.join(output_dir, f"phase_composite_mjo_{exp}.pdf")
        analyzer.plot_phase_composites(comps,figname)
